In [7]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
!ls "/content/drive/MyDrive/Potato Leaf Disease Dataset in Uncontrolled Environment"


Bacteria  Fungi  Healthy  Nematode  Pest  Phytopthora  Virus


In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model


In [10]:
import os
import shutil
import random

SOURCE_DIR = "/content/drive/MyDrive/Potato Leaf Disease Dataset in Uncontrolled Environment"
DEST_DIR = "/content/potato_dataset_split"

SPLITS = {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
}

os.makedirs(DEST_DIR, exist_ok=True)

for class_name in os.listdir(SOURCE_DIR):
    class_path = os.path.join(SOURCE_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    total = len(images)
    train_end = int(SPLITS["train"] * total)
    val_end = train_end + int(SPLITS["val"] * total)

    split_map = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split, files in split_map.items():
        split_class_dir = os.path.join(DEST_DIR, split, class_name)
        os.makedirs(split_class_dir, exist_ok=True)

        for file in files:
            shutil.copy(
                os.path.join(class_path, file),
                os.path.join(split_class_dir, file)
            )

print("✅ Dataset successfully split")


✅ Dataset successfully split


In [12]:
!ls /content/potato_dataset_split/


test  train  val


In [13]:
TRAIN_DIR = "/content/potato_dataset_split/train"
VAL_DIR   = "/content/potato_dataset_split/val"
TEST_DIR  = "/content/potato_dataset_split/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16   # safe + stable
EPOCHS = 10


In [14]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_data = val_test_gen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_data = val_test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


Found 2153 images belonging to 7 classes.
Found 459 images belonging to 7 classes.
Found 469 images belonging to 7 classes.


In [15]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [16]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(7, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)


In [17]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [18]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 95s 575ms/step - accuracy: 0.3370 - loss: 1.7907 - val_accuracy: 0.6122 - val_loss: 1.1356
Epoch 2/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 51s 379ms/step - accuracy: 0.5246 - loss: 1.2631 - val_accuracy: 0.6580 - val_loss: 0.9522
Epoch 3/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 50s 371ms/step - accuracy: 0.5608 - loss: 1.1525 - val_accuracy: 0.6492 - val_loss: 0.9147
Epoch 4/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 49s 366ms/step - accuracy: 0.6047 - loss: 1.0440 - val_accuracy: 0.6906 - val_loss: 0.8438
Epoch 5/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 50s 370ms/step - accuracy: 0.6325 - loss: 0.9394 - val_accuracy: 0.6885 - val_loss: 0.8357
Epoch 6/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 51s 377ms/step - accuracy: 0.6315 - loss: 0.9498 - val_accuracy: 0.7102 - val_loss: 0.8021
Epoch 7/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 51s 379ms/step - accuracy: 0.6590 - loss: 0.8902 - val_accuracy: 0.7037 - val_loss: 0.7956
Epoch 8/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 50s 372ms/step - accuracy: 0.6791 - loss: 0

In [19]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False


In [20]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [21]:
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)


Epoch 1/5
135/135 ━━━━━━━━━━━━━━━━━━━━ 82s 494ms/step - accuracy: 0.4765 - loss: 1.5566 - val_accuracy: 0.7255 - val_loss: 0.8281
Epoch 2/5
135/135 ━━━━━━━━━━━━━━━━━━━━ 52s 382ms/step - accuracy: 0.5847 - loss: 1.0824 - val_accuracy: 0.7277 - val_loss: 0.8151
Epoch 3/5
135/135 ━━━━━━━━━━━━━━━━━━━━ 52s 387ms/step - accuracy: 0.6306 - loss: 0.9948 - val_accuracy: 0.7298 - val_loss: 0.7937
Epoch 4/5
135/135 ━━━━━━━━━━━━━━━━━━━━ 51s 379ms/step - accuracy: 0.6504 - loss: 0.9474 - val_accuracy: 0.7124 - val_loss: 0.8181
Epoch 5/5
135/135 ━━━━━━━━━━━━━━━━━━━━ 52s 384ms/step - accuracy: 0.6746 - loss: 0.8823 - val_accuracy: 0.7146 - val_loss: 0.7882


In [22]:
test_loss, test_acc = model.evaluate(test_data)
print("✅ Final Test Accuracy:", test_acc)


30/30 ━━━━━━━━━━━━━━━━━━━━ 15s 502ms/step - accuracy: 0.7328 - loss: 0.7173
✅ Final Test Accuracy: 0.6652451753616333


In [24]:
model.save("/content/drive/MyDrive/potato_disease_mobilenetv2_final.keras")
